In [1]:
import os
import pandas as pd
import numpy as np
from Bio import SeqIO
from Bio.Seq import Seq
# --------------------------------
from scipy.stats import mannwhitneyu, ttest_ind, pearsonr
from scipy.cluster.hierarchy import linkage, leaves_list
import statsmodels.api as sm
from statsmodels.stats.multitest import multipletests
from statsmodels.formula.api import ols
# --------------------------------
from tqdm.notebook import tqdm
# ----------------------------------------------------------------
from lets_plot import *
LetsPlot.setup_html()
base_size = 14
tsne_plot_width = 650
theme_settings = theme(
    axis_text=element_text(size=base_size),
    axis_title=element_text(size=base_size * 1.2),
    legend_title=element_text(size=base_size * 1.2),
    legend_text=element_text(size=base_size * 0.9),
    plot_title=element_text(size=base_size * 1.4),
    axis_ticks_y=element_line(color='black', size=0.5),
    axis_line_y=element_line(color='black', size=0.5)
)
# ----------------------------------------------------------------
# from pathlib import Path
# import sys
# PROJECT_ROOT = Path("/lustre/groups/crna01/projects/collabs/gabi")
# if str(PROJECT_ROOT) not in sys.path:
#     sys.path.insert(0, str(PROJECT_ROOT))

In [2]:
# quick fasta loader(s) ------------------
from typing import Optional, Dict
def load_fasta_to_dict(
    fasta_path: str,
    ids_keep: Optional[set] = None,
    id_func=lambda x: x,
) -> Dict[str, str]:
    """Reads FASTA into {id: sequence}. Applies id_func to header token."""
    seqs: Dict[str, str] = {}
    with open(fasta_path, 'r') as fh:
        cur_id, cur_seq = None, []
        for line in fh:
            line = line.strip()
            if not line:
                continue
            if line.startswith('>'):
                if cur_id is not None:
                    seq = ''.join(cur_seq).upper()
                    if ids_keep is None or cur_id in ids_keep:
                        seqs[cur_id] = seq
                cur_id = id_func(line[1:].split()[0])
                cur_seq = []
            else:
                cur_seq.append(line)
        if cur_id is not None:
            seq = ''.join(cur_seq).upper()
            if ids_keep is None or cur_id in ids_keep:
                seqs[cur_id] = seq
    return seqs
## solely to load the RNAfold output
import re
def parse_dotbracket_file(path):
    records = []
    with open(path) as f:
        lines = [line.strip() for line in f if line.strip()]
    for i in range(0, len(lines), 3):
        header = lines[i]
        seq = lines[i+1]
        struct_line = lines[i+2]

        seq_id = header.lstrip(">")

        m = re.search(r"\((-?\d+(?:\.\d+)?)\)$", struct_line)
        energy = float(m.group(1)) if m else None

        records.append({"id": seq_id,
                        "sequence": seq,
                        "structure": struct_line,
                        "MFE": energy})
    return pd.DataFrame(records)

In [3]:
# function to bootstrap deltas

def bootstrap_deltas(
    data: pd.DataFrame,
    class_var: str,
    subgrp_var: str,
    ab: tuple,
    xy: tuple,
    value_col: str = "n_peaks_lnorm",
    n: int = 2000,
    random_state: int | None = None,
):
    """
    Returns raw deltas and Cohen's d for each class separately across bootstrap resamples.
    """
    rng = np.random.default_rng(random_state)
    dA_raw = np.empty(n)
    dB_raw = np.empty(n)
    dA_std = np.empty(n)
    dB_std = np.empty(n)

    for i in range(n):
        sample = data.sample(frac=1.0, replace=True, random_state=int(rng.integers(1e9)))

        grouped = sample.groupby([class_var, subgrp_var])[value_col]
        means = grouped.mean().unstack()
        stds  = grouped.std(ddof=1).unstack()
        ns    = grouped.count().unstack()

        dA = means.loc[ab[0], xy[0]] - means.loc[ab[0], xy[1]]
        dB = means.loc[ab[1], xy[0]] - means.loc[ab[1], xy[1]]

        nA0, nA1 = ns.loc[ab[0], xy[0]], ns.loc[ab[0], xy[1]]
        sA0, sA1 = stds.loc[ab[0], xy[0]], stds.loc[ab[0], xy[1]]
        nB0, nB1 = ns.loc[ab[1], xy[0]], ns.loc[ab[1], xy[1]]
        sB0, sB1 = stds.loc[ab[1], xy[0]], stds.loc[ab[1], xy[1]]

        def pooled_sd(n0, n1, s0, s1):
            df = (n0 - 1) + (n1 - 1)
            if df <= 0:
                return np.nan
            pv = ((n0 - 1) * (s0 ** 2) + (n1 - 1) * (s1 ** 2)) / df
            return np.sqrt(pv)

        sdA = pooled_sd(nA0, nA1, sA0, sA1)
        sdB = pooled_sd(nB0, nB1, sB0, sB1)

        dA_raw[i] = dA
        dB_raw[i] = dB
        dA_std[i] = dA / sdA if sdA and np.isfinite(sdA) and sdA > 0 else np.nan
        dB_std[i] = dB / sdB if sdB and np.isfinite(sdB) and sdB > 0 else np.nan

    return pd.DataFrame({
        "delta_A": dA_raw,
        "delta_B": dB_raw,
        "d_A": dA_std,
        "d_B": dB_std,
        "delta_diff": dA_raw - dB_raw
    })

##
def bootstrap_cohens_d(data, class_var, subgrp_var, ab, xy, n=2000, random_state=None):
    rng = np.random.default_rng(random_state)
    ds = []
    for _ in range(n):
        sample = data.sample(frac=1, replace=True)
        grouped = sample.groupby([class_var,subgrp_var])["n_peaks_lnorm"]
        means = grouped.mean().unstack()
        stds  = grouped.std().unstack()
        ns    = grouped.count().unstack()

        #(means['Retained', 'speckle'] - means['Retained', 'non-speckle']) - (means['Spliced', 'speckle'] - means['Spliced', 'non-speckle'])
        delta = (means.loc[ab[0],xy[0]] - means.loc[ab[0],xy[1]]) - \
                (means.loc[ab[1],xy[0]] - means.loc[ab[1],xy[1]])

        pooled_var = 0
        total_n = 0
        for i in [ab[0], ab[1]]:
            for j in [xy[0], xy[1]]:
                n = ns.loc[i,j]
                pooled_var += (n-1) * stds.loc[i,j]**2
                total_n += n
        pooled_var /= (total_n - 4)
        pooled_sd = np.sqrt(pooled_var)

        ds.append(delta / pooled_sd)

    return np.array(ds)

In [4]:
# functions to perform anove and delta enrichment on encore peaks

def prepare_encore_peaks_by_class(
    tab_meta: pd.DataFrame,
    tab_pks_enc: pd.DataFrame,
    rbp_speckle: list[str],
    class_col: str,
    class_names: list[str],
    class_assigner,
    meta_cols: list[str],
    dropna_subset: list[str],
):
    tab_enc_meta = tab_meta[meta_cols].merge(
        tab_pks_enc,
        left_index=True,
        right_index=True,
        how='inner'
    )
    tab_cls = tab_enc_meta.dropna(subset=dropna_subset).copy()
    tab_cls[class_col] = class_assigner(tab_cls)
    tab_cls = tab_cls[tab_cls[class_col].isin(class_names)].copy()

    tab_cls = tab_cls.reset_index().rename(columns={'index': 'EVENT'})
    tab_cls = tab_cls.melt(
        id_vars=['EVENT', 'LENGTH', class_col],
        value_vars=[col for col in tab_pks_enc.columns if col.endswith(('_K562', '_HepG2'))],
        var_name='rbp',
        value_name='n_peaks'
    )

    tab_cls['n_peaks_lnorm'] = tab_cls['n_peaks'] / tab_cls['LENGTH']
    tab_cls['rbp_base'] = tab_cls['rbp'].str.replace(r'_(K562|HepG2)$', '', regex=True)
    tab_cls['speckle_rbp'] = np.where(tab_cls['rbp_base'].isin(rbp_speckle), 'speckle', 'non-speckle')
    return tab_cls


def run_delta_bootstrap_anova(
    tab_long: pd.DataFrame,
    class_col: str,
    class_names: list[str],
    plot_filename: str,
    stats_filename: str,
    x_label: str,
    fill_map: dict[str, str],
):
    dat_sampled = tab_long.groupby([class_col, 'speckle_rbp']).sample(20000, random_state=42)
    model = ols(f'n_peaks_lnorm ~ C({class_col}) * C(speckle_rbp)', data=dat_sampled).fit()
    anova_table = sm.stats.anova_lm(model, typ=2)

    # Attach sample-size metadata for reporting.
    anova_table['n_total'] = len(dat_sampled)
    class_counts = dat_sampled[class_col].value_counts()
    for cname in class_names:
        col_name = f"n_{str(cname).replace('-', '_').replace(' ', '_')}"
        anova_table[col_name] = int(class_counts.get(cname, 0))
    speck_counts = dat_sampled['speckle_rbp'].value_counts()
    anova_table['n_speckle'] = int(speck_counts.get('speckle', 0))
    anova_table['n_non_speckle'] = int(speck_counts.get('non-speckle', 0))

    res = bootstrap_deltas(
        dat_sampled,
        class_col,
        'speckle_rbp',
        (class_names[0], class_names[1]),
        ('speckle', 'non-speckle')
    )
    df_bstrap = res[['delta_A', 'delta_B']].copy()
    df_bstrap.columns = class_names
    df_bstrap_long = df_bstrap.melt(
        value_vars=class_names,
        var_name=class_col,
        value_name='delta'
    )
    df_bstrap_long['delta'] = df_bstrap_long['delta'] * 1000

    gg_delta = ggplot(df_bstrap_long) +\
        geom_boxplot(aes(x=class_col, y='delta', fill=class_col, alpha=class_col), width=.75) +\
        labs(y='Δ (speckle - non-speckle) [n peaks per 1Kb]', x=x_label) +\
        scale_fill_manual(values=fill_map) +\
        scale_alpha_manual(values={name: .7 for name in class_names}) +\
        ggsize(150, 450) +\
        coord_cartesian(ylim=[0.05, 0.55]) +\
        theme_settings +\
        theme(legend_position='none')
    ggsave(gg_delta, filename=plot_filename, path=dir_plotting)
    anova_table.to_csv(os.path.join(dir_plotting, stats_filename), index=True)

    # Source data export for this delta boxplot panel.
    source_filename = f"{os.path.splitext(plot_filename)[0]}_source.csv"
    df_bstrap_long.to_csv(os.path.join(dir_plotting, source_filename), index=False)

    return dat_sampled, anova_table


def run_speckle_boxplot_tests(
    dat_sampled: pd.DataFrame,
    class_col: str,
    x_label: str,
    plot_filename: str,
    stats_filename: str,
    plot_fill_map: dict[str, str] | None = None,
):
    rows = []
    for cls, dfc in dat_sampled.groupby(class_col):
        a = dfc.loc[dfc['speckle_rbp'] == 'non-speckle', 'n_peaks_lnorm'].sample(5000)
        b = dfc.loc[dfc['speckle_rbp'] == 'speckle', 'n_peaks_lnorm'].sample(5000)
        if len(a) >= 1 and len(b) >= 1:
            stat_t, p_t = ttest_ind(a, b, equal_var=False)
        else:
            stat_t, p_t = np.nan, np.nan
        rows.append({
            'comparison': 'Speckle vs non-speckle',
            'group_X': 'non-speckle',
            'group_Y': 'speckle',
            'n_X': len(a),
            'n_Y': len(b),
            'n_total': len(a) + len(b),
            'value': 'RBP peaks / length',
            'group': cls,
            'test': "Welch's t-test",
            'statistic': stat_t,
            'p_value': p_t
        })
    tests = pd.DataFrame(rows).sort_values('group')
    tests['p_adj'] = multipletests(tests['p_value'], method='fdr_bh')[1]

    if plot_fill_map is None:
        plot_fill_map = {'non-speckle': '#635c6c', 'speckle': '#ad2021'}

    dat_plot = dat_sampled.assign(n_peaks_lnorm=lambda _df: _df.n_peaks_lnorm * 1000)
    gg_speck = ggplot(data=dat_plot) +\
        geom_boxplot(aes(x=class_col, y='n_peaks_lnorm', fill='speckle_rbp', alpha='speckle_rbp'), tooltips='none') +\
        scale_fill_manual(values=plot_fill_map) +\
        scale_alpha_manual(values={'non-speckle': .7, 'speckle': .7}) +\
        labs(y='Normalized peaks [n peaks per 1Kb]', x=x_label, fill='RBP group:', alpha='RBP group:') +\
        coord_cartesian(ylim=[0.006, 70]) +\
        scale_y_log10() +\
        ggsize(300, 450) +\
        theme_settings
    ggsave(gg_speck, filename=plot_filename, path=dir_plotting)
    tests.to_csv(os.path.join(dir_plotting, stats_filename), index=False)

    # Source data export for this boxplot panel.
    source_filename = f"{os.path.splitext(plot_filename)[0]}_source.csv"
    dat_plot.to_csv(os.path.join(dir_plotting, source_filename), index=False)

    return tests

In [5]:
# functions to estimate motif enrichment via GLM

## ---------------------------------
import re, math, numpy as np, pandas as pd
import statsmodels.api as sm
import statsmodels.formula.api as smf

# G4 motifs per AM
motifs = {
    "G4_3t_short": r'G{3,}[ACGT]{1,7}G{3,}[ACGT]{1,7}G{3,}[ACGT]{1,7}G{3,}',
    "G4_3t_long":  r'G{3,}[ACGT]{8,12}G{3,}[ACGT]{8,12}G{3,}[ACGT]{8,12}G{3,}',
    "G4_2t_short": r'G{2,}[ACGT]{1,7}G{2,}[ACGT]{1,7}G{2,}[ACGT]{1,7}G{2,}',
    "G4_2t_long":  r'G{2,}[ACGT]{8,12}G{2,}[ACGT]{8,12}G{2,}[ACGT]{8,12}G{2,}',
}


def compile_patterns(name_to_regex, overlap=True, ignore_case=True):
    flags = re.IGNORECASE if ignore_case else 0
    if overlap:
        # wrap with lookahead to capture overlapping matches
        comp = {name: re.compile(rf'(?=({pat}))', flags) for name, pat in name_to_regex.items()}
    else:
        comp = {name: re.compile(pat, flags) for name, pat in name_to_regex.items()}
    return comp

def iter_matches(seq, cregex):
    for m in cregex.finditer(seq):
        s = m.start(1) if m.lastindex else m.start()
        e = s + len(m.group(1) if m.lastindex else m.group(0))
        yield s, e

def gc_fraction(seq):
    s = seq.upper()
    if not s: return 0.0
    g = s.count('G'); c = s.count('C')
    return (g + c) / len(s)

# ---------- Build per-sequence feature table ----------
def summarize_sequences(seq_dict, group_map, compiled_patterns):
    rows = []
    for sid, seq in seq_dict.items():
        L = len(seq)
        if L == 0: continue
        gcf = gc_fraction(seq)
        grp = group_map[sid] if group_map is not None else "NA"
        for name, crex in compiled_patterns.items():
            # count overlapping hits
            cnt = sum(1 for _ in iter_matches(seq, crex))
            rows.append(dict(sequence_id=sid, group=grp, motif=name, length=L, gc=gcf, count=cnt))
    return pd.DataFrame(rows)

# ---------- Fit GLM: count ~ group with loglen offset ----------
def fit_glm_enrichment(df, family="poisson"):
    out = []
    for motif, sub in df.groupby("motif", sort=False):
        sub = sub.copy()
        sub["log_len"] = np.log(sub["length"])
        formula = f"count ~ C(group)"
        if family == "poisson":
            model = smf.glm(formula, data=sub, family=sm.families.Poisson(), offset=sub["log_len"])
            res = model.fit(cov_type="HC3")
        elif family == "nb":
            model = smf.glm(formula, data=sub, family=sm.families.NegativeBinomial(), offset=sub["log_len"])
            res = model.fit()
        else:
            raise ValueError("family must be 'poisson' or 'nb'")

        group_counts = sub.groupby("group").size().to_dict()
        n_total = int(len(sub))

        # report rate ratios
        for p, est in res.params.items():
            if p.startswith("C(group)[T."):
                term_group = p.split("C(group)[T.", 1)[1][:-1]
                n_term = int(group_counts.get(term_group, 0))
                ref_group = np.nan
                n_ref = np.nan
                if len(group_counts) == 2 and term_group in group_counts:
                    ref_group = [g for g in group_counts if g != term_group][0]
                    n_ref = int(group_counts.get(ref_group, 0))

                rr = math.exp(est)
                lo, hi = np.exp(res.conf_int().loc[p].values)
                out.append(dict(
                    motif=motif,
                    term=p,
                    group_term=term_group,
                    group_ref=ref_group,
                    n_term=n_term,
                    n_ref=n_ref,
                    n_total=n_total,
                    rate_ratio=rr,
                    ci_low=lo,
                    ci_high=hi,
                    pvalue=res.pvalues[p],
                    family=family
                ))
    return pd.DataFrame(out)

In [6]:
# functions to compute motif meta-coverage profiles

def _bin_edges(a: float, b: float, nbins: int) -> np.ndarray:
    return np.linspace(a, b, nbins + 1, dtype=float)

def _merge_intervals(starts, ends):
    if len(starts) == 0: return []
    order = np.argsort(starts)
    s = np.array(starts)[order]; e = np.array(ends)[order]
    out = []; cs, ce = s[0], e[0]
    for i in range(1, len(s)):
        if s[i] <= ce:
            ce = max(ce, e[i])
        else:
            out.append((cs, ce)); cs, ce = s[i], e[i]
    out.append((cs, ce))
    return out

def _covered_bp_per_bin(intervals, edges):
    # intervals: list of (s,e) in the region's local coords
    cov = np.zeros(len(edges)-1, dtype=float)
    if not intervals: return cov
    for (s,e) in intervals:
        s = float(s); e = float(e)
        if e <= edges[0] or s >= edges[-1]: continue
        i0 = np.searchsorted(edges, s, side='right') - 1
        i1 = np.searchsorted(edges, e, side='left')
        i0 = max(i0, 0); i1 = min(i1, len(edges)-1)
        for i in range(i0, i1):
            lo = max(edges[i], s); hi = min(edges[i+1], e)
            if hi > lo: cov[i] += (hi - lo)
    return cov

def _runs_per_bin(runs, edges):
    # runs: list[(s,e)], count if a run touches bin i
    cnt = np.zeros(len(edges)-1, dtype=float)
    for (s,e) in runs:
        i0 = np.searchsorted(edges, s, side='right') - 1
        i1 = np.searchsorted(edges, e, side='left')
        i0 = max(i0, 0); i1 = min(i1, len(edges)-1)
        for i in range(i0, i1):
            cnt[i] += 1.0
    return cnt

# ---------- Build a hits table from regex scanning ----------
def scan_regex_hits(seq_dict, compiled_patterns, group_map=None):
    """
    Returns a tidy DataFrame: sequence_id, motif, run_start, run_length, seq_len, group
    """
    recs = []
    for sid, seq in seq_dict.items():
        L = len(seq)
        grp = group_map[sid] if group_map is not None else "NA"
        for motif, crex in compiled_patterns.items():
            for s,e in iter_matches(seq, crex):
                recs.append(dict(sequence_id=sid, motif=motif,
                                 run_start=s, run_length=(e-s), seq_len=L, group=grp))
    df = pd.DataFrame(recs)
    if df.empty:
        return pd.DataFrame(columns=["sequence_id","motif","run_start","run_length","seq_len","group"])
    return df

# ---------- Meta-coverage for regex motifs ----------
def metacoverage_regex(
    hits_df,
    seq_lookup,                # dict or callable id->sequence
    group_col='group',
    label_col='motif',
    flank_len=50,
    nbins_left=None,
    nbins_core=100,
    nbins_right=None,
    per_seq_mode='any',        # 'any'|'count'|'length'|'frac'|'binary_fraction'
    label_order=None,
    group_order=None
):
    if nbins_left  is None: nbins_left  = flank_len
    if nbins_right is None: nbins_right = flank_len

    total_len = nbins_left + nbins_core + nbins_right
    off_left, off_core, off_right = 0, nbins_left, nbins_left + nbins_core
    labels = hits_df[label_col].unique() if label_order is None else list(label_order)
    groups = hits_df[group_col].unique() if group_order is None else list(group_order)

    use_union_bins = per_seq_mode in ('frac', 'binary_fraction')
    frames = []

    for g in groups:
        sub_g = hits_df[hits_df[group_col] == g]
        if sub_g.empty:
            continue
        nseqs_group = max(1, sub_g['sequence_id'].nunique())

        for lab in labels:
            sub_lab = sub_g[sub_g[label_col] == lab]
            acc = np.zeros(total_len, dtype=float)

            # iterate sequences to give each sequence equal weight
            for sid, gseq in sub_lab.groupby('sequence_id', sort=False):
                # pull sequence length (from table) and/or actual string for 'frac'
                L = int(gseq['seq_len'].iloc[0])
                s = gseq['run_start'].to_numpy(np.int64, copy=False)
                rl = gseq['run_length'].to_numpy(np.int64, copy=False)
                e = s + rl

                seq_cov = np.zeros(total_len, dtype=float)

                # 'frac' denominator: #nt in each bin (computed from actual sequence if available)
                denom = None
                if per_seq_mode == 'frac':
                    seq_str = seq_lookup(sid) if callable(seq_lookup) else seq_lookup[sid]
                    seq_b = np.frombuffer(seq_str.encode('ascii'), dtype=np.uint8)

                    # For 'frac', denominator is simply bin length in nt (independent of nucleotide),
                    # because regex hits are not nucleotide-specific like polyN.
                    bin_len = np.concatenate([
                        np.full(nbins_left,  flank_len, dtype=float) if nbins_left  > 0 else np.zeros(0),
                        np.full(nbins_core,  max(1, L - 2*flank_len), dtype=float) if nbins_core  > 0 else np.zeros(0),
                        np.full(nbins_right, flank_len, dtype=float) if nbins_right > 0 else np.zeros(0),
                    ])
                    denom = bin_len

                # LEFT region [0, flank_len)
                if flank_len > 0 and nbins_left > 0:
                    ls = np.clip(s, 0, flank_len); le = np.clip(e, 0, flank_len)
                    valid = le > ls
                    if np.any(valid):
                        intervals = _merge_intervals(ls[valid], le[valid]) if use_union_bins else list(zip(ls[valid], le[valid]))
                        edges = _bin_edges(0.0, float(flank_len), nbins_left)
                        if per_seq_mode == 'any':
                            # any: 1 if any run touches the bin
                            touch = _runs_per_bin(intervals if not use_union_bins else intervals, edges)
                            seq_cov[off_left:off_left+nbins_left] = (touch > 0).astype(float)
                        elif per_seq_mode == 'count':
                            seq_cov[off_left:off_left+nbins_left] = _runs_per_bin(intervals, edges)
                        elif per_seq_mode == 'length':
                            seq_cov[off_left:off_left+nbins_left] = _covered_bp_per_bin(intervals if use_union_bins else list(zip(ls[valid], le[valid])), edges)
                        elif per_seq_mode in ('frac','binary_fraction'):
                            vals = _covered_bp_per_bin(intervals, edges) if per_seq_mode=='frac' else _runs_per_bin(intervals, edges)
                            seq_cov[off_left:off_left+nbins_left] = vals

                # RIGHT region [L-50, L)
                if flank_len > 0 and nbins_right > 0:
                    right0 = L - flank_len
                    rs = np.clip(s, right0, L) - right0
                    re = np.clip(e, right0, L) - right0
                    valid = re > rs
                    if np.any(valid):
                        intervals = _merge_intervals(rs[valid], re[valid]) if use_union_bins else list(zip(rs[valid], re[valid]))
                        edges = _bin_edges(0.0, float(flank_len), nbins_right)
                        if per_seq_mode == 'any':
                            touch = _runs_per_bin(intervals if not use_union_bins else intervals, edges)
                            seq_cov[off_right:off_right+nbins_right] = (touch > 0).astype(float)
                        elif per_seq_mode == 'count':
                            seq_cov[off_right:off_right+nbins_right] = _runs_per_bin(intervals, edges)
                        elif per_seq_mode == 'length':
                            seq_cov[off_right:off_right+nbins_right] = _covered_bp_per_bin(intervals if use_union_bins else list(zip(rs[valid], re[valid])), edges)
                        elif per_seq_mode in ('frac','binary_fraction'):
                            vals = _covered_bp_per_bin(intervals, edges) if per_seq_mode=='frac' else _runs_per_bin(intervals, edges)
                            seq_cov[off_right:off_right+nbins_right] = vals

                # CORE region [50, L-50) rescaled
                core0, core1 = flank_len, L - flank_len
                core_len_abs = max(0, core1 - core0)
                if core_len_abs > 0 and nbins_core > 0:
                    cs = np.clip(s, core0, core1) - core0
                    ce = np.clip(e, core0, core1) - core0
                    valid = ce > cs
                    if np.any(valid):
                        intervals = _merge_intervals(cs[valid], ce[valid]) if use_union_bins else list(zip(cs[valid], ce[valid]))
                        edges = _bin_edges(0.0, float(core_len_abs), nbins_core)
                        if per_seq_mode == 'any':
                            touch = _runs_per_bin(intervals if not use_union_bins else intervals, edges)
                            seq_cov[off_core:off_core+nbins_core] = (touch > 0).astype(float)
                        elif per_seq_mode == 'count':
                            seq_cov[off_core:off_core+nbins_core] = _runs_per_bin(intervals, edges)
                        elif per_seq_mode == 'length':
                            seq_cov[off_core:off_core+nbins_core] = _covered_bp_per_bin(intervals if use_union_bins else list(zip(cs[valid], ce[valid])), edges)
                        elif per_seq_mode in ('frac','binary_fraction'):
                            vals = _covered_bp_per_bin(intervals, edges) if per_seq_mode=='frac' else _runs_per_bin(intervals, edges)
                            seq_cov[off_core:off_core+nbins_core] = vals

                # normalise for 'frac'
                if per_seq_mode == 'frac':
                    with np.errstate(divide='ignore', invalid='ignore'):
                        seq_cov = np.where(denom > 0, seq_cov / denom, 0.0)

                acc += seq_cov

            avg = acc / nseqs_group  # equal weight per sequence
            frames.append(pd.DataFrame({
                "bin": np.arange(total_len, dtype=int),
                "coverage": avg,
                label_col: lab,
                "group": g
            }))

    long_df = pd.concat(frames, ignore_index=True) if frames else \
              pd.DataFrame(columns=['bin','coverage',label_col,'group'])

    # annotate regions
    if not long_df.empty:
        L0, L1 = 0, nbins_left - 1
        C0, C1 = nbins_left, nbins_left + nbins_core - 1
        R0, R1 = nbins_left + nbins_core, total_len - 1
        b = long_df['bin'].to_numpy()
        region = np.where((b >= L0) & (b <= L1), 'left',
                 np.where((b >= C0) & (b <= C1), 'core', 'right'))
        long_df['region'] = pd.Categorical(region, ['left','core','right'], ordered=True)

    return long_df, dict(
        left_bins=(0, nbins_left-1),
        core_bins=(nbins_left, nbins_left+nbins_core-1),
        right_bins=(nbins_left+nbins_core, nbins_left+nbins_core+nbins_right-1),
        flank_len_bp=flank_len, nbins_left=nbins_left, nbins_core=nbins_core, nbins_right=nbins_right
    )


In [7]:
# function to prepare a heatmap

import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib.ticker import FixedLocator
#plt.rcParams['svg.fonttype'] = 'none'
plt.rcParams['pdf.use14corefonts'] = True

def plot_rbp_heatmap(
        tab_input, group_var,
        cell_line=('HepG2', 'K562'), row_order=None, x_lab=None, log_vals=False, rows_to_highlight=None,
        hm_cmap='viridis', gridline_w=.1, fig_size=(4, 32), show_plot=True, file_name=None
    ):
    
    if x_lab is None:
        x_lab = group_var

    value_cols = [col for col in tab_input.columns if col.endswith(cell_line) and col != 'SBDS_K562']
    tab_zscored = tab_input.copy()
    if log_vals:
        tab_zscored[value_cols] = np.log1p(tab_zscored[value_cols] + 1e-6) 
    tab_zscored[value_cols] = (tab_input[value_cols] - tab_input[value_cols].mean()) / (tab_input[value_cols].std() + 1e-6)
    ##
    group_means = tab_zscored.groupby(group_var, observed=False)[value_cols].mean()
    heatmap_data = group_means.T
    ## determine row order
    if row_order is None and len(group_means) == 2:
        group_labels = list(group_means.index)
        delta = group_means.loc[group_labels[0]] - group_means.loc[group_labels[1]]
        row_order = delta.sort_values(ascending=False).index
        heatmap_data = heatmap_data.loc[row_order]
    elif row_order is None:
        linkage_rows = linkage(heatmap_data, method='average', metric='euclidean')
        row_order = heatmap_data.index[leaves_list(linkage_rows)]
        heatmap_data = heatmap_data.loc[row_order]
    else:
        heatmap_data = heatmap_data.loc[row_order]
    if len(cell_line) < 2:
        heatmap_data.index = [name.rsplit('_', 1)[0] for name in heatmap_data.index]
    ##
    plt.figure(figsize=fig_size)
    ax = sns.heatmap(
        heatmap_data,
        cmap=hm_cmap,         
        linewidths=gridline_w,           # thicker lines for bigger gaps
        linecolor='white',  
        cbar_kws={'shrink': 0.05, 'label': 'RBP peaks per Kb [z-score]'}
    )
    ax.set_xticklabels(ax.get_xticklabels(), fontsize=8, rotation=45, ha='right')

    ax.yaxis.set_major_locator(FixedLocator(np.arange(len(heatmap_data.index)) + 0.5))
    ax.set_yticklabels(heatmap_data.index, fontsize=6, rotation=0)
    ##
    # if rows_to_highlight:
    #     row_color = [rbp for rbp in value_cols if rbp.startswith(rows_to_highlight)]
    #     for label in ax.get_yticklabels():
    #         if label.get_text() in row_color:
    #             label.set_color('red')

    if rows_to_highlight:
        highlight_set = set(rows_to_highlight)  # e.g., rbp_speckle base names
        for label in ax.get_yticklabels():
            base_name = label.get_text().rsplit('_', 1)[0]
            if base_name in highlight_set:
                label.set_color('red')

    plt.subplots_adjust(left=0.25)
    #plt.title("Parnet peaks [z-score]", fontsize=12)
    plt.xlabel(x_lab, fontsize=10)
    plt.ylabel("RBP", fontsize=10)
    ##
    if file_name:
        plt.savefig(f'{file_name}', format='pdf', bbox_inches='tight')
    if show_plot:
        plt.show()
    else:
        plt.close()

In [8]:
# function to prepare a scatter plot
import matplotlib.pyplot as plt
import seaborn as sns

def plot_scatter_colored(tab_data, x_name, y_name, color_col,
                         x_lab=None, y_lab=None,
                         cmap='viridis', fig_size=(8, 8),
                         point_size=10, alpha=0.8,
                         vmin=None, vmax=None,
                         file_name=None, dpi=300, 
                         show_plot=True):
    """
    Plot a simple scatter plot colored by a continuous variable, with optional clipping.

    Parameters
    ----------
    tab_data : pandas.DataFrame
        Input table containing x, y, and color columns.
    x_name, y_name, color_col : str
        Column names for x-axis, y-axis, and color mapping.
    cmap : str
        Matplotlib colormap name for the continuous variable.
    vmin, vmax : float or None
        Minimum and maximum values to clip the color variable.
    """
    plt.figure(figsize=fig_size)

    # Clip color values if requested
    colors = tab_data[color_col].copy()
    if vmin is not None:
        colors = np.maximum(colors, vmin)
    if vmax is not None:
        colors = np.minimum(colors, vmax)

    sc = plt.scatter(
        tab_data[x_name],
        tab_data[y_name],
        c=colors,
        cmap=cmap,
        s=point_size,
        alpha=alpha,
        edgecolors='none',
        rasterized=True,
        vmin=vmin,
        vmax=vmax
    )

    plt.colorbar(sc, label=color_col)

    plt.xlabel(x_lab if x_lab else x_name, fontsize=10)
    plt.ylabel(y_lab if y_lab else y_name, fontsize=10)

    if file_name:
        plt.savefig(file_name, format='pdf', bbox_inches='tight', dpi=dpi)
    if show_plot:
        plt.show()
    else:
        plt.close()

In [9]:
## paths & load data
dir_data = os.path.join('resubmission', 'data')
dir_results = os.path.join('resubmission', 'results')
dir_plotting = os.path.join(dir_results, 'plots')
dir_models = os.path.join(dir_results, 'models')
path_metadata = os.path.join(dir_data, 'metadata_selected.csv')

tab_meta = pd.read_csv(path_metadata, index_col=0)

# cutoffs
#high_cut, low_cut = -0.364098, -0.651847 # HL scaled&revised calss boundaries
high_cut, low_cut  = -0.364742, -1.342584
pir_cut = 10

## shared ENCORE inputs
tab_pks_enc = pd.read_csv(os.path.join(dir_data, 'external', 'ENCORE', 'peaks_encore.csv'))
tab_pks_enc = tab_pks_enc[~tab_pks_enc['intron'].duplicated(keep=False)]
tab_pks_enc.index = tab_pks_enc['intron']
tab_pks_enc = tab_pks_enc.drop(columns=['intron'])
tab_pks_enc.index.name = 'EVENT'

tab_speckle_rbps = pd.read_csv(
    os.path.join(dir_data, 'external', 'ENCORE', 'HepG2-HeLa_FeaturesMatrix.csv')
)
rbp_speckle = tab_speckle_rbps.query('Quality > 1').dropna(subset=['Speckles'])['RBP'].to_list()

---
---

In [11]:
## Supplementary Figure 4 A - histogram of PIRs split into classes

tmp_ir_hist = tab_meta.dropna(subset=['PIR_Nuc_baseline']).copy()
class_conds = [
    tmp_ir_hist['PIR_Nuc_baseline'] >= 30,
    tmp_ir_hist['PIR_Nuc_baseline'] < 1
]
class_names = ['Retained', 'Spliced']
tmp_ir_hist['class_ir'] = np.select(class_conds, class_names, default='else')

gg_hist_ir = ggplot(data=tmp_ir_hist) +\
    geom_histogram(aes(x='PIR_Nuc_baseline', fill='class_ir', alpha='class_ir'), color='black', bins=30) +\
    geom_vline(xintercept=30, color='#2a1f3f', linetype='dashed') + \
    geom_vline(xintercept=1, color='#a76794', linetype='dashed') +\
    scale_fill_manual(values={'Spliced': '#a76794', 'Retained': '#2a1f3f', 'else': 'grey'}) +\
    scale_alpha_manual(values={'Spliced': .7, 'Retained': .7, 'else': .3}) +\
    ggsize(650, 350) +\
    labs(
        fill='Intron\nRetention\ngroup:', alpha='Intron\nRetention\ngroup:',
        x='Nuclear PIR (%)', y='Introns [n / 1000]'
    ) +\
    theme_settings
ggsave(gg_hist_ir, filename='SF4A_hist_classSplit_ir.svg', path=dir_plotting)

# Source data export for Figure SF4A
tmp_ir_hist.to_csv(os.path.join(dir_plotting, 'SF4A_hist_classSplit_ir_source.csv'))


---

In [ ]:
## Supplementary Figure 4 B - Boxplots of PIR for stability classes

tmp_boxes = tab_meta.query('PIR_Nuc_baseline > @pir_cut').dropna(subset=['PIR_Nuc_baseline', 'hl_gated_intwise_scaled_logged']).copy()
class_conds = [
    tmp_boxes['hl_gated_intwise_scaled_logged'] > high_cut,
    tmp_boxes['hl_gated_intwise_scaled_logged'] < low_cut
]
class_names = ['LL-RIs', 'SL-RIs']
tmp_boxes['class_hls'] = np.select(class_conds, class_names, default='else')
tmp_boxes = tmp_boxes[tmp_boxes.class_hls != 'else']

binder = []
a = tmp_boxes.loc[tmp_boxes['class_hls'] == 'LL-RIs', 'PIR_Nuc_baseline'].dropna()
b = tmp_boxes.loc[tmp_boxes['class_hls'] == 'SL-RIs', 'PIR_Nuc_baseline'].dropna()
n_ll, n_sl = len(a), len(b)
stat_t, p_t = ttest_ind(a, b, equal_var=False)
test_res = pd.DataFrame(
    {
        'comparison': 'LL-RIs vs SL-RIs',
        'group_X': 'LL-RIs',
        'group_Y': 'SL-RIs',
        'n_X': n_ll,
        'n_Y': n_sl,
        'n_total': n_ll + n_sl,
        'value': 'PIR',
        'group': '-',
        'test': ["two sided Welch's t-test"],
        'stat': [stat_t],
        'p_value': [p_t]
    }
)

gg_ir_hls = ggplot(tmp_boxes) +\
    geom_boxplot(aes(x='class_hls', y='PIR_Nuc_baseline', fill='class_hls', alpha='class_hls'), width=.7, outlier_size=0.7, outlier_alpha=0.35) +\
    geom_hline(yintercept=0, linetype='dashed', color='red') +\
    labs(y='Intron retention [PIR]', x='IR HL group') +\
    scale_fill_manual(values={'LL-RIs': '#2a1f3f', 'SL-RIs': '#a76794'}) +\
    scale_alpha_manual(values={'LL-RIs': .7, 'SL-RIs': .7})+\
    coord_cartesian(ylim=[0, 100]) +\
    ggsize(150, 450) +\
    theme_settings +\
    theme(legend_position='none')
ggsave(gg_ir_hls, filename='SF4B_boxes_pir_hls.svg', path=dir_plotting)

# Source data export for Supplementary Fig4B
tmp_boxes.to_csv(os.path.join(dir_plotting, 'SF4B_boxes_pir_hls_source.csv'))

##
test_res['p_adj'] = multipletests(test_res['p_value'], method='fdr_bh')[1]
test_res.to_csv(os.path.join(dir_plotting, 'SF4B_boxes_pir_hls_stats.csv'), index=False)


---

In [16]:
## Figure 3 F - delta Encode peaks speckle enrichment in LL-RIs vs SL-RIs

## class_hls analysis (Fig3D)
tab_enc_meta_hls = prepare_encore_peaks_by_class(
    tab_meta=tab_meta,
    tab_pks_enc=tab_pks_enc,
    rbp_speckle=rbp_speckle,
    class_col='class_hls',
    class_names=['LL-RIs', 'SL-RIs'],
    class_assigner=lambda df: np.select(
        [
            df['hl_gated_intwise_scaled_logged'] > high_cut,
            df['hl_gated_intwise_scaled_logged'] < low_cut
        ],
        ['LL-RIs', 'SL-RIs'],
        default='else'
    ),
    meta_cols=['LENGTH', 'PIR_Nuc_baseline', 'hl_gated_intwise_scaled_logged', 'Speckle_Enrichment'],
    dropna_subset=['PIR_Nuc_baseline', 'hl_gated_intwise_scaled_logged']
)
tab_enc_meta_hls['n_peaks_lnorm'] = tab_enc_meta_hls['n_peaks_lnorm'] * 1000 # make per 1kb
tab_enc_meta_hls['n_peaks_lnorm_logged'] = np.log10(tab_enc_meta_hls['n_peaks_lnorm'] + 1)

dat_sampled_hls, anova_hls = run_delta_bootstrap_anova(
    tab_long=tab_enc_meta_hls,
    class_col='class_hls',
    class_names=['LL-RIs', 'SL-RIs'],
    plot_filename='F3F_anova_boxes_deltaspeck_hls.svg',
    stats_filename='F3F_anova_boxes_deltaspeck_hls.csv',
    x_label='IR HL group',
    fill_map={'LL-RIs': '#2a1f3f', 'SL-RIs': '#a76794'}
)


## class_ir analysis
tab_enc_meta_ir = prepare_encore_peaks_by_class(
    tab_meta=tab_meta,
    tab_pks_enc=tab_pks_enc,
    rbp_speckle=rbp_speckle,
    class_col='class_ir',
    class_names=['Retained', 'Spliced'],
    class_assigner=lambda df: np.select(
        [
            df['PIR_Nuc_baseline'] >= 30,
            df['PIR_Nuc_baseline'] < 1
        ],
        ['Retained', 'Spliced'],
        default='else'
    ),
    meta_cols=['LENGTH', 'PIR_Nuc_baseline', 'Speckle_Enrichment'],
    dropna_subset=['PIR_Nuc_baseline']
)

dat_sampled_ir, anova_ir = run_delta_bootstrap_anova(
    tab_long=tab_enc_meta_ir,
    class_col='class_ir',
    class_names=['Retained', 'Spliced'],
    plot_filename='F3F_anova_boxes_deltaspeck_ir.svg',
    stats_filename='F3F_anova_boxes_deltaspeck_ir.csv',
    x_label='IR group',
    fill_map={'Retained': '#2a1f3f', 'Spliced': '#a76794'}
)

In [19]:
## Supplementary Figure 4 C - Boxplots of RBP peaks for speckle vs non-speckle RBPs, split by IR stability class
tests_hls = run_speckle_boxplot_tests(
    dat_sampled=dat_sampled_hls,
    class_col='class_hls',
    x_label='IR group',
    plot_filename='SF4C_boxes_speck_hls.svg',
    stats_filename='SF4C_boxes_speck_hls_stats.csv',
)
dat_sampled_hls.to_csv(os.path.join(dir_plotting, 'SF4C_boxes_speck_hls_source.csv'), index=False)
tests_ir = run_speckle_boxplot_tests(
    dat_sampled=dat_sampled_ir,
    class_col='class_ir',
    x_label='IR group',
    plot_filename='SF4C_boxes_speck_ir.svg',
    stats_filename='SF4C_boxes_speck_ir_stats.csv',
)
dat_sampled_ir.to_csv(os.path.join(dir_plotting, 'SF4C_boxes_speck_ir_source.csv'), index=False)

---

In [ ]:
## Supplementary Figure 4 G

----

In [20]:
## Supplementary Figure 4 E - UMAP projection of ENCORE speckle RBP peaks
##
tab_umap_hlrev = pd.read_csv(
    os.path.join(
        dir_results, 'models', 'HL_revised', 'parnet_512_jc_50percgap_256bp_PIR10',
        'umap_embeddings.csv'
    ),
    index_col=0
)

speckle_cols = [c for c in tab_pks_enc.columns if any(c.startswith(f"{rbp}_") for rbp in rbp_speckle)]
tab_pks_enc_speckle = tab_pks_enc[speckle_cols].copy()
tab_pks_enc_speckle['n_specklerbp_peaks_sum'] = tab_pks_enc_speckle[speckle_cols].sum(axis=1)
tab_pks_enc_speckle['n_specklerbp_peaks_mean'] = tab_pks_enc_speckle[speckle_cols].mean(axis=1)
##
test_speckle_umap = tmp_boxes.merge(
    tab_pks_enc_speckle[['n_specklerbp_peaks_sum', 'n_specklerbp_peaks_mean']],
    left_index=True, right_index=True, how='left'
).merge(
    tab_umap_hlrev,
    left_index=True, right_index=True, how='left'
)
## save for reporting in SF4
tab_meta_rbp = tab_meta.merge(
    tab_pks_enc_speckle[['n_specklerbp_peaks_sum', 'n_specklerbp_peaks_mean']],
    left_index=True, right_index=True, how='left'
)
for col in ['n_specklerbp_peaks_sum', 'n_specklerbp_peaks_mean']:
    tab_meta_rbp[f'{col}_lnorm'] = 1000 * tab_meta_rbp[col] / tab_meta_rbp['LENGTH']
    tab_meta_rbp[f'{col}_lnorm_logged'] = np.log10(tab_meta_rbp[f'{col}_lnorm'] + 1)
    test_speckle_umap[f'{col}_lnorm'] = 1000 * test_speckle_umap[col] / test_speckle_umap['LENGTH']
    test_speckle_umap[f'{col}_lnorm_logged'] = np.log10(test_speckle_umap[f'{col}_lnorm'] + 1)
tab_meta_rbp.to_csv(os.path.join(dir_data, 'metadata_selected_encodespeckleRBPs.csv'))

# Unified UMAP source table for Supplementary Fig4E
test_speckle_umap.to_csv(os.path.join(dir_plotting, 'SF4E_umap_specklerbp_peaks_source.csv'))
##
plot_scatter_colored(
    test_speckle_umap,
    x_name='UMAP1_HLrev', y_name='UMAP2_HLrev',
    color_col='n_specklerbp_peaks_mean_lnorm_logged',
    x_lab='UMAP1', y_lab='UMAP2', vmax=.5,
    cmap=sns.cubehelix_palette(as_cmap=True, rot=.4, start=-.05),
    fig_size=(6.5,5.25), point_size=10, 
    show_plot=False,
    file_name=os.path.join(dir_plotting, 'SF4E_umap_specklerbp_peaks.svg')
)

---

In [ ]:
## Supplementary Fig3 C - Encode peaks speckle enrichment in LL-RIs and SL-RIs


In [ ]:
## Supplementary Fig3 C - Encode peaks speckle enrichment in Retained and Spliced


In [ ]:
## Supplementary Fig3 E - Encode peaks heatmap for LL-RIs and SL-RIs

tab_pks_enc_meta = tab_meta[['PIR_Nuc_baseline', 'hl_gated_intwise_scaled_logged', 'Speckle_Enrichment']].merge(
    tab_pks_enc,
    left_index=True,
    right_index=True
)

hmtest_enc = tab_pks_enc_meta.dropna(subset='hl_gated_intwise_scaled_logged').copy()
class_conds = [
    hmtest_enc['hl_gated_intwise_scaled_logged'] > high_cut,
    hmtest_enc['hl_gated_intwise_scaled_logged'] < low_cut
]
class_names = ['LL-RIs', 'SL-RIs']
hmtest_enc['class_hls'] = np.select(class_conds, class_names, default='nan')
hmtest_enc = hmtest_enc[hmtest_enc['class_hls'] != 'nan']
## heatmaps for ENCORE peaks by half-life classes
for cl in ['HepG2', 'K562']:
    tab_hm_source = hmtest_enc[['class_hls'] + [col for col in hmtest_enc.columns if col.endswith((cl))]].copy()
    plot_rbp_heatmap(
        tab_input=tab_hm_source,
        group_var='class_hls',
        cell_line=(cl,),
        hm_cmap=sns.cubehelix_palette(as_cmap=True, rot=.4, start=-.05),
        x_lab='Intron stability group',
        rows_to_highlight=tuple(rbp_speckle),
        fig_size=(1, 18),
        show_plot=False,
        file_name=f'{dir_plotting}/SF3E_hm_encore_hls_{cl}.pdf'
    )
    tab_hm_source.to_csv(os.path.join(dir_plotting, f'SF3E_hm_encore_hls_{cl}_source.csv'))
    

In [ ]:
## Supplementary Fig3 F - G-quad enrichment in LL-RIs

### G-quad enrichment in IR HL groups
tmp_hls_gquad = tab_meta.query('PIR_Nuc_baseline > @pir_cut').dropna(subset=['PIR_Nuc_baseline', 'hl_gated_intwise_scaled_logged']).copy()
class_conds = [
    tmp_hls_gquad['hl_gated_intwise_scaled_logged'] > high_cut,
    tmp_hls_gquad['hl_gated_intwise_scaled_logged'] < low_cut
]
class_names = ['LL-RIs', 'SL-RIs']
tmp_hls_gquad['class_hls'] = np.select(class_conds, class_names, default='else')
tmp_hls_gquad = tmp_hls_gquad[tmp_hls_gquad.class_hls != 'else']
tmp_hls_gquad = tmp_hls_gquad.reset_index()
##
seq_to_class = {}
id_idx = tmp_hls_gquad.columns.get_loc('EVENT') + 1
class_idx = tmp_hls_gquad.columns.get_loc('class_hls') + 1
for rowtuple in tmp_hls_gquad.itertuples():
    seq_to_class[rowtuple[id_idx]] = rowtuple[class_idx] # append class
##
seq_dict = load_fasta_to_dict(
    fasta_path=os.path.join(dir_data, 'introns_50fks.fasta'),
    ids_keep=tmp_hls_gquad.EVENT.to_list(),
    id_func=lambda x: x.split('_')[0]
)
compiled = compile_patterns(motifs, overlap=True)
df_feat = summarize_sequences(seq_dict=seq_dict, group_map=seq_to_class, compiled_patterns=compiled)
df_feat['count_lnorm'] = 1000 * df_feat['count'] / df_feat['length']
df_feat['count_lnorm_logged'] = np.log1p(df_feat['count_lnorm'])
##
result = fit_glm_enrichment(df_feat, family="poisson")
result['inverse_lgrr'] = np.log2(1 / result['rate_ratio'])
result['inverse_lgci_high'] = np.log2(1 / result['ci_low'])
result['inverse_lgci_low'] = np.log2(1 / result['ci_high'])
##
gg_GqaudEnrcih_ir = ggplot(result) +\
    geom_point(aes(x='motif', y='inverse_lgrr')) +\
    geom_errorbar(aes(x='motif', ymin='inverse_lgci_low', ymax='inverse_lgci_high')) +\
    geom_hline(yintercept=0, color='red') +\
    labs(y='G-quad motif hits\nlog2FC(LL-RIs - SL-RIs)', x='G-quad motif') +\
    coord_cartesian(ylim=[0, 2.4]) +\
    theme_settings +\
    ggsize(250, 500)
ggsave(gg_GqaudEnrcih_ir, filename='SF3F_dots_GquadEnrich_hls.svg', path=dir_plotting)

# Source data export for Supplementary Fig3F
result.to_csv(os.path.join(dir_plotting, 'SF3F_dots_GquadEnrich_hls_source.csv'), index=False)

result.to_csv(os.path.join(dir_plotting, 'SF3F_dots_GquadEnrich_hls_stats.csv'), index=False)

In [ ]:
## Supplementary Fig3 G - meta-profiles of G-quad density in LL-RIs and SL-RIs

## G-quad metaprofiles in IR HL groups
hits = scan_regex_hits(seq_dict, compiled, group_map=seq_to_class)
meta_df, meta_info = metacoverage_regex(
    hits_df=hits,
    seq_lookup=seq_dict,
    group_col='group',
    label_col='motif',
    flank_len=50,
    nbins_left=20,
    nbins_core=100,
    nbins_right=20,
    per_seq_mode='binary_fraction'
)
##
meta_df_2t = meta_df[meta_df.motif.isin(['G4_2t_short', 'G4_2t_long'])].copy()
gg_MetaGquad2T_hls = ggplot(meta_df_2t) +\
    geom_line(aes('bin', 'coverage', color='group', alpha='group'), size=1.5) +\
    scale_color_manual(values={'SL-RIs': '#a76794', 'LL-RIs': '#2a1f3f'}) +\
    scale_alpha_manual(values={'SL-RIs': .7, 'LL-RIs': .7}) +\
    geom_vline(xintercept=20) +\
    geom_vline(xintercept=120) +\
    facet_wrap('motif', nrow=1) +\
    labs(y='G-quad motif profile', x='Intron body bin', color='IR HL group:', alpha='IR HL group:') +\
    theme_settings +\
    ggsize(1200, 350)
ggsave(gg_MetaGquad2T_hls, filename='SF3G_metaprofile_Gquad2T_hls.svg', path=dir_plotting)
meta_df_2t.to_csv(os.path.join(dir_plotting, 'SF3G_metaprofile_Gquad2T_hls_source.csv'), index=False)

##
meta_df_3t = meta_df[meta_df.motif.isin(['G4_3t_short', 'G4_3t_long'])].copy()
gg_MetaGquad3T_hls = ggplot(meta_df_3t) +\
    geom_line(aes('bin', 'coverage', color='group', alpha='group'), size=1.5) +\
    scale_color_manual(values={'SL-RIs': '#a76794', 'LL-RIs': '#2a1f3f'}) +\
    scale_alpha_manual(values={'SL-RIs': .7, 'LL-RIs': .7}) +\
    geom_vline(xintercept=20) +\
    geom_vline(xintercept=120) +\
    facet_wrap('motif', nrow=1) +\
    labs(y='G-quad motif profile', x='Intron body bin', color='IR HL group:', alpha='IR HL group:') +\
    theme_settings +\
    ggsize(1200, 350)
ggsave(gg_MetaGquad3T_hls, filename='SF3G_metaprofile_Gquad3T_hls.svg', path=dir_plotting)
meta_df_3t.to_csv(os.path.join(dir_plotting, 'SF3G_metaprofile_Gquad3T_hls_source.csv'), index=False)

'/ictstr01/groups/crna01/projects/collabs/gabi/resubmission/results/plots/SF3G_metaprofile_Gquad3T_hls.svg'